# Trabajo final · Data Scientist · Alzo

**Mantenimiento de ascensores** · Empresa ficticia. Los datos son sintéticos y **se generan por código en este notebook**: ningún resultado representa una situación real.

## La pregunta oficial

> **¿A qué ascensores mandamos al técnico esta semana?**

**Decisión que habilita:** La lista semanal de visitas, con capacidad para 60 equipos.

**Tipo de problema:** Clasificación muy desbalanceada sobre series de tiempo.

Las instrucciones completas (qué se entrega, criterios de calidad y defensa) están en el PDF **«Trabajo final · Data Scientist · Instrucciones»**. Este notebook es tu lugar de trabajo: vas de los datos sucios a la recomendación sin salir de acá.

| Sección | Módulo del programa |
| --- | --- |
| 1 | Introducción: definición del problema |
| 2 | Python: limpieza |
| 3 | Estadística |
| 4 | Machine Learning y regresión |
| 5 | Deep Learning |
| 6 | Apache Superset |
| 7 | De modelo a producción |
| 8 | Ciencia de datos aplicada al negocio |
| 9 | Fichas y recomendación |

## 0 · Preparación

Ejecutá esta celda una sola vez. `xgboost` es opcional: si querés usarlo, descomentá la última línea.

In [ ]:
%pip install -q numpy pandas scikit-learn matplotlib
# %pip install -q xgboost


## 1 · Datos y definición del problema

La celda siguiente **genera las tablas**. Tarda unos segundos. Después mirá qué hay con `inventario()` y `diccionario()`.

Antes de modelar, definí el problema: **tipo de problema, población, período, granularidad y variable objetivo**, y qué información está realmente disponible en el momento de decidir.

In [ ]:
import zlib, base64
TEXTO = (
    "eNqVWd1u2zgWvs9TCBp0IbWOKst2mjrrARYTBFggk5udm4FhCIxFO+xYooaS3Wa6mXfZB9ir3u1tXmy/Q1IS5Z9OUqS2RJ7/P55D"
    "i7yUqvaKbV4+eqzyinLglazI8Ii/Mjs7y/jKW/OCK6bSjNWyCiqei82GzZI4uZgMvB1XGctmN2xT8XB65uGf8magFCkQknkECmy7"
    "qVNVrBvc8Mq7A8zHOL7yrvGQfBhdeVmMpzKLfhE5r2qWl4EPFpPzeIg/HyiZgEwzgntHcLVMa4BmfFOzAOwY+K15cB0OvG0h6pl/"
    "7YdanFwCRgJTRcsHKZY8mPs/D/2B5/+c6M+Rvxh4d9B8No/GAy8aQa0omSzAk0M3o81yI8pARWuW5ywYRfHAG9PHHdjFUTLwRhOA"
    "l6KSlQNPVpDbIusQAUhIof4ex2HEqvqx5IEoauBvqz05Fa9ExoulYBsSVa7EUhSsoucHWZWiNutLmXOlgTpNJkYLfMSX+BgmRh9B"
    "JAwX8ORrrqpgCFEutTLaYMWW75grByiOsK14uWFLbn195Ymiqo1Dzg8cEmjLvfVGF5Oejo5zWgKw0ucHrjiZS1SicLxJ5tXi0HcZ"
    "IQI58bgYN8FxcT6c+NgkUj1/vYRiHEFzQjF6b1gO3DiK43gygexA4l/KAAtjetW035G36S1owmqmIyg0O7ShnThzHER7Kiqkytkm"
    "iIkt+QZBQNBDjRgPCVWHj5FFFCLdiYy9XJnhBZGGDVYMKUZROF9oUiupPAGCXoNjslR7Wnu5lKKqZBFA/7lYkFDX8GgjAZbCsMUQ"
    "K6+YelmVGsEqlI+gjZNOMAcZ+g2hLuVlJf7gs5w0wHufBd6Gk3A/xijrNSt8zkn9ouJgSJEgVqsAq6H3o0dBDt1/UVseLlr938EA"
    "gaDIqIMvsDWZ4QuZAVjGMDem4Fyzmt0olvPAYA68pdxs86KazX1B2YXC41Py3LOKp7teObBOHVIFiKNRUw8uBl4SJQ1KfQxljKQc"
    "G/gRHi/Ic0um1ibxTLUAmWFCMF2gTNwIa8uBddBO3BtWf3CFSk15e002rPPy6DovK83NDc0hhHEBkj2IIVW9FoCpo4SZSo6st9FY"
    "d9E4pJ0pEObTgVcvdP5dQEe7QGFBEQRBDICmfQCb7ANbEMcqxnm0fCcLTmBMNZYxXurvJRS3kS0DlHWJeSyF+W6PG++9LnJzQh14"
    "UyfnBnRkkap/wPE3kYh2bLPliK8bhC+zb1De837wSsWXW4V04pU3HHvZ83/pDEaBrtrU+82Y1DLO2ZeAyEPnMeW9YQaZYf7PgAx+"
    "w1YLEZKYw/EVGWMOwX5bUH7Al1Dms/f2LR7H2hrd5qS3abRCUspq72Sg/NrwIrihKIXFyP+rrH/AJLFJeCoEBGsIQShSHalXQtnf"
    "t9wrpFdzhQrBCo8XJpNP2NOQGIAVLNg3TWeW15skji5Pm2Sk46Fnkp1gn3jVK6M6i7tgalN3+LJg+hB2sUSpVKLibRWzbYVpJyzX"
    "t+CK4xTGyVGCLqnoXJjkHIb7h86ozdrQVKLY5ONSKiU4HAX6F85ZZFm0NYey6Hw/iw5Pto6NpZ6JtWy8o9hjMF/5/7i7Pv8qyDHT"
    "eJw9+Ycn1MJg89+3otQB1yvTX31DNjX7/tSyQak2hzJWzANWRJY2TQ+Wm0ds6MMWS/ob7yu+fGAp9REMJ5CQBe3prqbpOgLdZAC0"
    "BtcURRgQ+HwywqLHhaA30RouKu8fA5wdoc5zylQq7sWRfToRadNqOvfNEZRSc+MvjOL/4nBQFQA/jBQXRca/9FqAMFqhqS5Y0O8m"
    "bX1xaWsNIafIWaoZaRbB0Y66YwyEU4ybgtb2dM2SLRcv9dy8q46L1hVGwunxht8toQ7/xjkNrtNK57KWio7zcouMMj30Uha1kqaF"
    "ZvcbrhdXiheIjIUpVTehaacTaqJHCfXU1E3T/0Wo23DkZlrCKBkxPOj4EzovPza0qE8Ln3ST1vhD8bWoIIX2xYpFjvJH9O537ZcO"
    "3f7Q84P3/E1hcrCukC9qB2kr67aosjv8Yt2lgNO1KfrOFvWzH1H5UYplxMqSwwCBda2AIRsX/tLqkbHHapaRAXHy7VB/xE76Tp9J"
    "oqSdKF11vXwtk0MtqAZq5yscusQ4zeFx1bCX+1Er3Y6wH8BtsC4VN0XDRqBuGHP2KNVf5wCc3ls6JDp9bWCMm3O2FxmNbNOe8iZX"
    "KCit5siKJauDOQTRGsC2Yg0D8VTXgJlutI2p5lRfJQWaDt+uOgyNADK05y8gM5xipE2qU6aB38sWajAnDa7OFoPLq6USpTbGwp2J"
    "ZEQa6W7YCSQoihcMS8/fCrRWcg3LU34OjiIeBALQ2adthTOxfcdcwkh8TRBONUYLm2TbAF8f0tS10HFYS4yTis6kR6/aYjQ3eSjQ"
    "S9R2Hsh59bBWItuf53pXGdrioljPfPFJj8wCyEJEiu34hk4OTa2umwXblOzIlX8G7chIfM2gGHp/8wIg/Z0GxlBTtCIJMSfEBb2a"
    "J03s9sVFXIi2eFPZhs7zuqYl9A7W9Xme0pmrWy7Nl7ZrnpepKc9T03C1W6YL0Rj04Oy0fVFq6jlg2iUHrG1vUqYFtW8NhD27OfWr"
    "TW9AelwRaK0bu/l+HTS3JWF4WAVNkzs5skU4oR1CnbJ2SfPqkolMuuMSBf8taNxLuemGJp6lqHH4X2gCRr6pxf43jnFSgrRIGxf/"
    "iLk57fydpcjFIrWNWYOm7D2d5RoCkFrAyRVANRDZgLTRL1G1zQOKGoqK2/mfenGB7qDitakOQaZk6ZSIFXuwM2Jzd/G9ywx3sPh4"
    "cOVE9cDMOlQWVghA12x31mJmZ/59UlRESEi0R0W6opOXcKg750wtH+heg2dB01fe9gt0M7rZBJn7qLAZy1IK5H55IuKIwhsqIT9R"
    "/gLYCfcjsLdRt4+++yMmggn8Rp2Hu6VPl4130n/gFG3kcg6gfoYRy5NbLj8bJpvjJsZGz8I02Z2y8DJnFU1ohPNaC1P03pqjj+LZ"
    "3vpN9JVw2+EafTSbw2rTKXwSAONcHGt9s2250fAVhtIND1YAnJE9B54xc4rxoOa6qWguszFPyfKRajFhz4/xp42ov97Mhf3xKU7M"
    "2UcIYZNnzpF8O9C0FmFPwO9Jp8/f7yUo3fDvmBL6siFAM106HWHVG0GqdsBAw9qpvd0Pw4okp7vFKgLgfDxduINJS1xxVOuiS4At"
    "Yrcc9EZF0Pz6xZkS9QVesQgHfSx9eTrYHzK9F6Hq8fYQ+ZyQR4fIVROa9uy/3W8Jyd+dPW/3Wzv4+IOpA+ag7JWAo8k8oezQwFFW"
    "k0FXehj13/x6/iY/f5NRX3ccIHv/Jn//5te2saVLOdm6TSbfF10mR2S/sIh7DSphAr6/ekpccy4kZtxppFn9hTSrY9JMLKI7MOox"
    "KnG75VNWcXH7U1iD3qyeomCaB9LEDtmtOryl7N4oEHHeEHc2TlnKpXMwt7eE3J3vU7LXI8dj7s6eHROdCyCsUDU/c6WP6MrkBng2"
    "NysZJfSMCoFtWk+ByPtP6I5tDOrfEMH+q28tRpcyKHp+00GnFS/0NSg2buknLzPFYlnt7CWOJHhzUUKTUfJ05lQTzeDs7PqfP/20"
    "x+awb/WXz9/0/RTE9Vi11Jx99xqpeTq4SPLbX9Iyiarq8ap+/o/vXCuZB32rieGCJgz/xCWTWYQMXruOKcPv3zPpZ4JpZWhtYC6L"
    "ps2rVsaoSNOI3mw594LIZV1tvef/6U072jwN2jp91DevsGYzEfh0q03MMMBakv6x6aBZoVmLaJleCLrk+Xt9Q9MbGPQbV4zIdeAE"
    "5nZl0+bV8neQfHfOsE+aEInrHx81ujWiZ5Z7OP25o3vtK8RICPB1jX0k4F9ra/fiwBgdvCgSK65/Y3r+RiYwY7tzJ9CEWLvRzvZT"
    "eqbZGqHHMrol4j0Ke6P91F6LEVit2D37pMPVHeKnzitFu2uBNrVfrXcb106kmbX9i0H6ybyUhXYJWYbWn7/5hxd6nSZ2qWXWHhmH"
    "RrZbIPj0dPZ/Fj5cLw=="
)
exec(zlib.decompress(base64.b64decode(TEXTO)).decode())

datos = generar_datos()          # genera las tablas, con una semilla fija: todos los equipos de esta empresa ven lo mismo
globals().update(datos)          # cada tabla queda disponible como variable


In [ ]:
SLUG = "05_Alzo"
import os, sqlite3
import pandas as pd

def inventario():
    """Tablas disponibles: filas, columnas y porcentaje de celdas faltantes."""
    return pd.DataFrame([(n, len(df), len(df.columns), round(float(df.isna().mean().mean() * 100), 1)) for n, df in datos.items()], columns=["tabla", "filas", "columnas", "% de faltantes"])

def diccionario(tabla=None):
    """Qué significa cada columna. Es una descripción neutra: no dice qué está limpio ni cuándo se conoce cada dato."""
    return pd.DataFrame([(t, c, d) for t, cols in DICC.items() if tabla in (None, t) for c, d in cols.items()], columns=["tabla", "columna", "qué es"])

bitacora_calidad = pd.DataFrame(columns=["problema", "dónde", "filas afectadas", "qué decidimos y por qué"])
def anotar(problema, donde, filas, decision):
    """Deja anotado un problema de calidad y la decisión tomada. Es parte de la trazabilidad."""
    bitacora_calidad.loc[len(bitacora_calidad)] = [problema, donde, filas, decision]; return bitacora_calidad

def exportar_superset(carpeta="superset", **tablas):
    """Guarda las tablas que pases como archivo SQLite y como CSV, para conectar Apache Superset."""
    os.makedirs(carpeta, exist_ok=True); con = sqlite3.connect(f"{carpeta}/{SLUG}.db")
    for nombre, df in tablas.items():
        df.to_csv(f"{carpeta}/{nombre}.csv", index=False); df.astype({c: "object" for c in df.columns if str(df[c].dtype).startswith("str")}).to_sql(nombre, con, index=False, if_exists="replace")
    con.close(); print(f"Listo: {carpeta}/{SLUG}.db y un CSV por tabla. Conectá Superset a ese archivo."); return carpeta


In [ ]:
inventario()

In [ ]:
diccionario()

### Tu definición del problema

Completá este diccionario. Si cambia más adelante, cambiá también la respuesta y anotá por qué.

In [ ]:
definicion = {
    "tipo de problema": "",
    "población": "",
    "período": "",
    "granularidad (una fila es…)": "",
    "variable objetivo": "",
    "qué se sabe en el momento de decidir": "",
}
definicion


## 2 · Python

**Qué tenés que resolver en tu caso:** Unificar los identificadores de equipo, las unidades de temperatura y las marcas de tiempo; decidir qué hacer con los días sin lectura.

Limpiá **sin borrar en silencio**: cada decisión se anota con `anotar(...)` y queda en la bitácora de calidad.

In [ ]:
# Tu trabajo


In [ ]:
bitacora_calidad

## 3 · Estadística

**Qué tenés que resolver en tu caso:** Distribución de las lecturas por equipo, valores atípicos y tasa de fallas en una ventana de 14 días.

Mostrá distribuciones e intervalos, y al menos una comparación o prueba con su conclusión y su límite.

In [ ]:
# Tu trabajo


## 4 · Modelado

**Qué tenés que resolver en tu caso:** Variables de ventana, regresión logística, Random Forest y XGBoost; validación por tiempo y fallas capturadas en las 60 visitas.

Empezá por una referencia simple. Validá sin mirar el futuro y elegí la métrica por el costo del error, no por costumbre.

In [ ]:
# Tu trabajo


## 5 · Deep Learning

**Qué tenés que resolver en tu caso:** Una red recurrente sobre las ventanas de sensores contra las variables agregadas: ¿vale la pena?

Entrená una red y compará contra la mejor referencia con la misma validación. Decidí si vale su costo y su opacidad.

In [ ]:
# Tu trabajo


## 6 · Superset

**Qué tenés que resolver en tu caso:** Tablero con los equipos priorizados, las fallas capturadas y el estado de los sensores.

Con `exportar_superset(nombre=tabla, ...)` llevás tus tablas a un archivo SQLite para conectar Apache Superset. Un gráfico por pregunta.

In [ ]:
# Tu trabajo


## 7 · Producción

**Qué tenés que resolver en tu caso:** Contrato de entrada del servicio, prueba de humo con equipos de borde y señales de deriva de los sensores.

Definí el contrato de entrada (campos, tipos y rangos), la versión, una prueba de humo y las señales de monitoreo con umbral, acción y responsable.

In [ ]:
# Tu trabajo


## 8 · Negocio

**Qué tenés que resolver en tu caso:** El costo de una visita en vano frente al de una falla; explicación de por qué un equipo quedó en la lista.

Decisión, costos, explicación global y local, y una recomendación proporcional a la evidencia.

In [ ]:
# Tu trabajo


## 9 · Fichas y recomendación

### Ficha de evidencia

| Campo | Tu respuesta |
| --- | --- |
| Decisión que se toma | |
| Población y período | |
| Procedimiento | |
| Resultado | |
| Validación | |
| Límite | |

### Ficha de modelo y plan de monitoreo

| Campo | Tu respuesta |
| --- | --- |
| Qué modelo se propone y por qué ese | |
| Con qué datos y qué versión | |
| Contrato de entrada | |
| Riesgos principales | |
| Señales a vigilar, umbral, acción y responsable | |

### Recomendación (una página)

- **Evidencia:** qué muestran los datos.
- **Interpretación:** qué creemos que significa, sin presentarlo como causalidad.
- **Recomendación:** qué conviene hacer, proporcional a la evidencia.
- **Una limitación y un siguiente paso.**